# 0. Import Libraries

In [15]:
import os, sys

def add_project_path(module_folder="model_implementation"):
    candidates = [
        os.path.abspath("."),   # current directory (Deepnote case)
        os.path.abspath("../src/")   # parent directory (local notebooks case)
    ]

    for path in candidates:
        if os.path.exists(os.path.join(path, module_folder)):
            if path not in sys.path:
                sys.path.append(path)
            return path

    raise ImportError(f"Could not find '{module_folder}' in current or parent directory")

add_project_path("model_implementation")
add_project_path("cnn")

'c:\\Users\\Aryo\\PersonalMade\\ITB Kuliah Semesteran\\Semester 6\\IF3270 Pembelajaran Mesin\\Tubes\\ML-Tubes-2_RecursiveLearnaholic\\src'

In [16]:
import subprocess
import sys
print(sys.executable)

subprocess.check_call([sys.executable, "-m", "pip", "install", "tensorflow==2.21.0", "scikit-learn", "pandas", "matplotlib"])

import tensorflow as tf
print("TensorFlow successfully imported:", tf.__version__)

c:\Users\Aryo\PersonalMade\ITB Kuliah Semesteran\Semester 6\IF3270 Pembelajaran Mesin\Tubes\ML-Tubes-2_RecursiveLearnaholic\venv\Scripts\python.exe
TensorFlow successfully imported: 2.21.0


In [17]:
from pathlib import Path
from datetime import datetime

import joblib
import itertools
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split

from model_implementation.ffnn import FFNN
from model_implementation.layer.loss import BCELoss
from cnn.layers import Conv2D, LocallyConnected2D, MaxPooling2D, AveragePooling2D, GlobalMaxPooling2D, GlobalAveragePooling2D, Flatten
from cnn.utility import image_loader, batch_loader, image_paths

# 1. Global Variables

In [19]:
SEED = 42
DEVICE = 'cpu'
IMAGE_SIZE = (150, 150) 
BATCH_SIZE = 32

plt.style.use("seaborn-v0_8")
np.random.seed(SEED)

CATEGORIES = {
    'buildings': 0, 
    'forest': 1,
    'glacier': 2,
    'mountain': 3,
    'sea': 4,
    'street': 5 
}
INV_CATEGORIES = {v: k for k, v in CATEGORIES.items()}

gpu_devices = tf.config.list_physical_devices('GPU')
if gpu_devices:
    print(f"GPU Detected: {gpu_devices}")
    for gpu in gpu_devices:
        tf.config.experimental.set_memory_growth(gpu, True)
    DEVICE = "/GPU:0" 
else:
    print("No GPU found, defaulting to CPU.")
    DEVICE = "/CPU:0"

No GPU found, defaulting to CPU.


# 2. Loading Data

In [20]:
def load_intel_dataset(root_path, target_size=(150, 150)):
    """
    Loads images and labels from the directory structure:
    root/label/image.jpg
    """
    X = []
    y = []
    
    root = Path(root_path)
    for category, label in CATEGORIES.items():
        cat_path = root / category
        if not cat_path.exists():
            continue
            
        print(f"Loading {category}...")
        paths = image_paths(cat_path)
        
        for p in paths:
            try:
                img = image_loader(p, target_size=target_size)
                X.append(img)
                y.append(label)
            except Exception as e:
                print(f"Error loading {p}: {e}")
                
    return np.array(X, dtype="float32"), np.array(y, dtype="int32")

TRAIN_DIR = Path("../data/seg_train/seg_train")
TEST_DIR = Path("../data/seg_test/seg_test")
PRED_DIR = Path("../data/seg_pred/seg_pred")

print("--- Loading Training Data ---")
X_train, y_train = load_intel_dataset(TRAIN_DIR, target_size=IMAGE_SIZE)

print("\n--- Loading Test Data ---")
X_test, y_test = load_intel_dataset(TEST_DIR, target_size=IMAGE_SIZE)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=SEED)

print(f"\nFinal Shapes:")
print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

--- Loading Training Data ---
Loading buildings...
Loading forest...
Loading glacier...
Loading mountain...
Loading sea...
Loading street...

--- Loading Test Data ---
Loading buildings...
Loading forest...
Loading glacier...
Loading mountain...
Loading sea...
Loading street...

Final Shapes:
Train: (11227, 150, 150, 3), Val: (2807, 150, 150, 3), Test: (3000, 150, 150, 3)


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import f1_score
import itertools
import pandas as pd
import numpy as np
import os
import pickle
import gc

BASE_SAVE_PATH = "../models/cnn"
os.makedirs(BASE_SAVE_PATH, exist_ok=True)

def get_generator(images, labels):
    def generator():
        for i in range(len(images)):
            yield images[i], labels[i]
    return generator

def prepare_dataset_safe(X, y, batch_size=32, shuffle=False):
    gen = get_generator(X, y)
    dataset = tf.data.Dataset.from_generator(
        gen,
        output_signature=(
            tf.TensorSpec(shape=(150, 150, 3), dtype=tf.float32),
            tf.TensorSpec(shape=(), dtype=tf.int64)
        )
    )
    if shuffle:
        dataset = dataset.shuffle(buffer_size=500)
    return dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)

train_steps = int(np.ceil(len(X_train) / 32))
val_steps = int(np.ceil(len(X_val) / 32))
train_ds = prepare_dataset_safe(X_train, y_train, batch_size=32, shuffle=True)
val_ds = prepare_dataset_safe(X_val, y_val, batch_size=32)
test_ds = prepare_dataset_safe(X_test, y_test, batch_size=32)

keras_variations = {
    'num_layers': [2, 3],        
    'filters': [32, 64],       
    'kernel_size': [3, 5],     
    'pooling_type': ['max', 'avg']  
}

keras_keys, keras_values = zip(*keras_variations.items())
keras_experiments_configs = [dict(zip(keras_keys, v)) for v in itertools.product(*keras_values)]

keras_results_registry = []

# 4. Training Loop
for i, keras_cfg in enumerate(keras_experiments_configs):
    print(f"\n--- Running Keras Experiment {i+1}/16: {keras_cfg} ---")

    tf.keras.backend.clear_session()
    keras_model = models.Sequential(name=f"Keras_Architecture_{i}")

    keras_model.add(layers.Conv2D(
        keras_cfg['filters'], 
        (keras_cfg['kernel_size'], keras_cfg['kernel_size']), 
        activation='relu', 
        padding='same',
        input_shape=(150, 150, 3)
    ))

    # Hidden Layers
    for _ in range(keras_cfg['num_layers'] - 1):
        keras_model.add(layers.Conv2D(
            keras_cfg['filters'], 
            (keras_cfg['kernel_size'], keras_cfg['kernel_size']), 
            activation='relu', 
            padding='same'
        ))

        if keras_cfg['pooling_type'] == 'max':
            keras_model.add(layers.MaxPooling2D((2, 2))) 
        else:
            keras_model.add(layers.AveragePooling2D((2, 2)))
            
    keras_model.add(layers.Flatten())
    keras_model.add(layers.Dense(6, activation='softmax'))

    keras_model.compile(
        optimizer='adam', 
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    keras_history = keras_model.fit(
        train_ds, 
        epochs=10, 
        steps_per_epoch=train_steps,
        validation_data=val_ds, 
        validation_steps=val_steps,
        verbose=1
    )

    # 5. Evaluation & Persistence
    keras_raw_preds = keras_model.predict(test_ds, verbose=0)
    keras_y_pred = np.argmax(keras_raw_preds, axis=1)
    keras_f1_macro = f1_score(y_test, keras_y_pred, average='macro')
    
    # Save weights
    weight_filename = f"keras_weights_exp_{i}.weights.h5"
    weight_full_path = os.path.join(BASE_SAVE_PATH, weight_filename)
    keras_model.save_weights(weight_full_path)
    
    # Log result
    result_entry = {
        'experiment_id': i,
        'config': keras_cfg, 
        'f1_macro': keras_f1_macro, 
        'history': keras_history.history,
        'keras_weight_path': weight_full_path
    }
    keras_results_registry.append(result_entry)
    
    # Backup registry immediately
    with open(os.path.join(BASE_SAVE_PATH, "keras_results_backup.pkl"), "wb") as f:
        pickle.dump(keras_results_registry, f)
        
    # Heavy cleanup
    del keras_model
    gc.collect()

# 6. Final Summary Save
df_keras_benchmarks = pd.DataFrame(keras_results_registry)
df_keras_benchmarks.to_csv(os.path.join(BASE_SAVE_PATH, "keras_experiment_summary.csv"), index=False)

print(f"\n[SUCCESS] All 16 Keras models and results are saved in: {BASE_SAVE_PATH}")


--- Running Keras Experiment 1/16: {'num_layers': 2, 'filters': 32, 'kernel_size': 3, 'pooling_type': 'max'} ---
Epoch 1/10
351/351 ━━━━━━━━━━━━━━━━━━━━ 46s 126ms/step - accuracy: 0.6370 - loss: 1.0352 - val_accuracy: 0.7325 - val_loss: 0.7586
Epoch 2/10


c:\Users\Aryo\PersonalMade\ITB Kuliah Semesteran\Semester 6\IF3270 Pembelajaran Mesin\Tubes\ML-Tubes-2_RecursiveLearnaholic\venv\Lib\site-packages\keras\src\trainers\epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()
